# Main Pipeline — Tổng hợp quy trình ML (Toàn bộ các giai đoạn)

> Notebook này tổng hợp toàn bộ luồng công việc của dự án, bao gồm thu thập dữ liệu, EDA, tiền xử lý, tạo đặc trưng, huấn luyện mô hình và đánh giá.

Mục tiêu:
- Chuẩn hóa luồng thực thi end-to-end để dễ tái tạo trên môi trường local hoặc Colab.
- Sử dụng các module trong thư mục `modules/` để tránh lặp lại logic.
- Ghi lại và lưu artifacts (EDA, features, models, báo cáo) vào thư mục `reports/` và `features/`.

Ghi chú:
- Chạy từng cell tuần tự hoặc dùng `Run all` sau khi đã cài đặt dependencies.
- Không mount dữ liệu cá nhân; sử dụng data trong `data/` hoặc nguồn công khai đã cấu hình.

## 1. Thiết lập và import

Thiết lập môi trường Python cơ bản và các import cần thiết cho notebook.

In [7]:
from pathlib import Path
import os
import sys
import warnings

import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f"Project root: {ROOT}")
print("Setup completed for full pipeline aggregation (Stages 1-5)")

Project root: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder
Setup completed for full pipeline aggregation (Stages 1-5)


## 2. Load Config va Modules

In [8]:
from config import CONFIG
from modules.data_loader import load_data_from_config, create_dataframe
from modules.eda import generate_eda_report, plot_missing_values
from modules.preprocessing import preprocess_pipeline

for folder in ["reports/eda", "reports/evaluation", "features", "data/processed"]:
    (ROOT / folder).mkdir(parents=True, exist_ok=True)

print("CONFIG loaded")
print("Target:", CONFIG["data"]["target_column"])

CONFIG loaded
Target: is_canceled


## 3. Điều phối pipeline

Các bước thực thi chính của pipeline: thu thập dữ liệu → EDA → tiền xử lý → tạo đặc trưng → huấn luyện → đánh giá.

### 3.1 Thu thập dữ liệu

Tải dữ liệu nguồn và chuyển về dataframe để phục vụ các bước xử lý tiếp theo.

In [9]:
data_cfg = CONFIG["data"]
target_col = data_cfg["target_column"]
local_file = ROOT / data_cfg["file_path"]

if local_file.exists():
    print(f"[Data] Use local cache: {local_file}")
    df, metadata = create_dataframe(str(local_file))
else:
    print("[Data] Local cache not found. Download from configured source...")
    df, metadata = load_data_from_config()

print(f"Data shape: {df.shape}")
print(f"Rows: {metadata['n_rows']} | Columns: {metadata['n_columns']} | Missing: {metadata['total_missing']}")
display(df.head())

[Data] Use local cache: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\data\hotel_bookings.csv
[Data Loader] Đọc dữ liệu thành công - shape: (119390, 32)
Data shape: (119390, 32)
Rows: 119390 | Columns: 32 | Missing: 129425


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [10]:
# 3.2 EDA
eda_output_dir = ROOT / CONFIG["eda"]["output_dir"]
eda_output_dir.mkdir(parents=True, exist_ok=True)

eda_report = generate_eda_report(df, output_dir=str(eda_output_dir))
missing_plot_path = eda_output_dir / "missing_values_chart.png"
_ = plot_missing_values(df, output_path=str(missing_plot_path))

print("EDA completed")
print("EDA report:", eda_output_dir / "eda_report.json")
print("Missing chart:", missing_plot_path)

missing_focus = eda_report["missing_values_analysis"]["target_columns_missing"]
print("Top missing columns:")
for col, val in sorted(missing_focus.items(), key=lambda x: x[1], reverse=True)[:10]:
    print(f"- {col}: {val}")

[EDA] Tạo báo cáo thống kê mô tả
[EDA] Đã lưu báo cáo thống kê mô tả tại: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\eda_report.json
[EDA] Vẽ biểu đồ missing values và lưu vào c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
[EDA] Đang vẽ biểu đồ missing values và lưu vào c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png...
[EDA] Hoàn tất lưu biểu đồ tại: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
EDA completed
EDA report: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\eda_report.json
Missing chart: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda\missing_values_chart.png
Top missing columns:
- company: 112593
- agent: 16340
- country: 488
- children: 4


In [11]:
# 3.3 Preprocessing
X, y, preprocessing_metadata = preprocess_pipeline(
    df=df,
    target_column=target_col,
    config=CONFIG["preprocessing"],
)

X = np.asarray(X)
y = np.asarray(y)

print("Preprocessing completed")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Imputation:", preprocessing_metadata["imputation"]["method"])
print("Encoding:", preprocessing_metadata["encoding"]["method"])
print("Scaling:", preprocessing_metadata["scaling"]["method"])

[Preprocessing] Bắt đầu chạy full preprocessing pipeline...

--- BƯỚC 1: XỬ LÝ MISSING DATA (IMPUTATION) ---
[Preprocessing] Đã xoá các cột do missing values quá cao: ['company', 'reservation_status', 'reservation_status_date']
[Preprocessing] Đã điền giá trị 0.0 cho missing values của cột 'agent'
[Preprocessing] Áp dụng SimpleImputer với strategy='median' cho numeric features
[Preprocessing] Đã xử lý xong. Số cột còn missing data: 0
[Preprocessing] Đã lưu dataset sau khi xử lý missing data tại: data/processed\dataset_after_imputation.csv

--- BƯỚC 2: ENCODING ---
[Preprocessing] Áp dụng OneHot encoding cho 10 categorical columns: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'customer_type']
[Preprocessing] OneHot encoding hoàn thành. Số features: 28 → 256

--- BƯỚC 3: SCALING ---
[Preprocessing] Áp dụng StandardScaler cho 256 numeric features
  - Mean range: [0.0000, 2016.1566]


### 3.4 Tạo đặc trưng (Giai đoạn 3)

Áp dụng các kỹ thuật tạo đặc trưng (ví dụ: PCA) và lưu artifacts đặc trưng vào thư mục `features/`.

In [12]:
from sklearn.model_selection import train_test_split
from modules.features import apply_pca, save_features, load_features
from pathlib import Path
import numpy as np

# Split data into train/test
test_size = CONFIG["models"]["train_test_split"]["test_size"]
random_state = CONFIG["models"]["train_test_split"]["random_state"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=test_size, random_state=random_state, stratify=y
)

print(f"[Features] Train/test split: {X_train.shape[0]}/{X_test.shape[0]} samples")

# Apply PCA if enabled
features_cfg = CONFIG["features"]
X_train_pca, pca_info = apply_pca(X_train, features_cfg)

# Transform test set using train PCA components
if pca_info["enabled"]:
    from sklearn.decomposition import PCA
    pca = PCA(n_components=pca_info["n_components"])
    pca.fit(X_train)
    X_test_pca = pca.transform(X_test)
    total_var = pca_info.get('cumulative_variance')[-1] if pca_info.get('cumulative_variance') else sum(pca_info.get('explained_variance_ratio', []))
    print(f"[Features] PCA: {X_train.shape[1]} → {X_train_pca.shape[1]} dims (variance: {float(total_var):.2%})")
else:
    X_train_pca, X_test_pca = X_train, X_test
    print("[Features] PCA disabled")

# Save features to disk
features_output_path = ROOT / features_cfg["output"]["path"]
save_features(X_train_pca, y_train, features_cfg["output"])
print(f"[Features] Artifacts saved to: {features_output_path}")

[Features] Train/test split: 95512/23878 samples
[Features] Áp dụng PCA với variance_threshold=0.95
[Features] PCA hoàn tất: 256 → 210 components
[Features] Tổng explained variance: 0.9506
[Features] PCA: 256 → 210 dims (variance: 95.06%)
[Features] Đã lưu X tại: features\processed_features_X.npy
[Features] Đã lưu y tại: features\processed_features_y.npy
[Features] Artifacts saved to: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\features\processed_features


### 3.5 Huấn luyện mô hình (Giai đoạn 4a)

Huấn luyện các mô hình truyền thống và lưu lại kết quả để so sánh.

In [13]:
from modules.models import (
    train_logistic_regression, train_svm, train_random_forest, train_mlp
)

models_cfg = CONFIG["models"]
models_cfg["mlp"]["enabled"] = True  # Bonus deep learning model for comparison
results = {}

print("[Models] Training all models...")

# Train Logistic Regression
if models_cfg.get("logistic_regression", {}).get("enabled", True):
    lr_model, lr_pred = train_logistic_regression(
        X_train_pca, y_train, X_test_pca, 
        params=models_cfg.get("logistic_regression", {}).get("params", {})
    )
    results["logistic_regression"] = {
        "model": lr_model,
        "predictions": lr_pred,
        "y_proba": lr_model.predict_proba(X_test_pca)[:, 1] if hasattr(lr_model, "predict_proba") else None
    }
    print("✓ Logistic Regression trained")

# Train SVM
if models_cfg.get("svm", {}).get("enabled", True):
    svm_model, svm_pred = train_svm(
        X_train_pca, y_train, X_test_pca,
        params=models_cfg.get("svm", {}).get("params", {})
    )
    results["svm"] = {
        "model": svm_model,
        "predictions": svm_pred,
        "y_proba": svm_model.predict_proba(X_test_pca)[:, 1] if hasattr(svm_model, "predict_proba") else None
    }
    print("✓ SVM trained")

# Train Random Forest
if models_cfg.get("random_forest", {}).get("enabled", True):
    rf_model, rf_pred = train_random_forest(
        X_train_pca, y_train, X_test_pca,
        params=models_cfg.get("random_forest", {}).get("params", {})
    )
    results["random_forest"] = {
        "model": rf_model,
        "predictions": rf_pred,
        "y_proba": rf_model.predict_proba(X_test_pca)[:, 1] if hasattr(rf_model, "predict_proba") else None
    }
    print("✓ Random Forest trained")

# Train MLP (Deep Learning Bonus)
if models_cfg.get("mlp", {}).get("enabled", False):
    try:
        mlp_model, mlp_pred = train_mlp(
            X_train_pca, y_train, X_test_pca,
            params=models_cfg.get("mlp", {}).get("params", {})
        )
        results["mlp"] = {
            "model": mlp_model,
            "predictions": mlp_pred,
            "y_proba": mlp_model.predict(X_test_pca).flatten() if hasattr(mlp_model, "predict") else None
        }
        print("✓ MLP (Deep Learning - bonus) trained")
    except Exception as e:
        print(f"✗ MLP training failed: {e}")

print(f"\n[Models] Total models trained: {len(results)}")

[Models] Training all models...
[Models] Huấn luyện Logistic Regression
✓ Logistic Regression trained
[Models] Huấn luyện SVM
✓ SVM trained
[Models] Huấn luyện Random Forest
✓ Random Forest trained
[Models] Huấn luyện MLP (TensorFlow/Keras)
747/747 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
✓ MLP (Deep Learning - bonus) trained

[Models] Total models trained: 4


### 3.6 Đánh giá và so sánh mô hình (Giai đoạn 4b)

Tính toán các chỉ số hiệu năng, sinh biểu đồ và tổng hợp bảng so sánh giữa các mô hình.

In [14]:
from modules.evaluation import evaluate_model, compare_models, compute_metrics

eval_cfg = CONFIG["evaluation"]
eval_output_dir = ROOT / eval_cfg["output_dir"]
eval_output_dir.mkdir(parents=True, exist_ok=True)

print("[Evaluation] Computing metrics and generating plots...")

evaluation_results = {}
for model_name, model_data in results.items():
    eval_result = evaluate_model(
        y_true=y_test,
        y_pred=model_data["predictions"],
        y_proba=model_data.get("y_proba"),
        model_name=model_name,
        output_dir=str(eval_output_dir),
        config=eval_cfg
    )
    evaluation_results[model_name] = eval_result
    print(f"✓ {model_name}: Acc={eval_result['accuracy']:.4f}, F1={eval_result['f1']:.4f}")

# Generate model comparison chart
comparison_path = compare_models(
    evaluation_results,
    output_path=str(eval_output_dir / "model_comparison.png")
)
print(f"\n✓ Model comparison chart saved to: {comparison_path}")

[Evaluation] Computing metrics and generating plots...
✓ logistic_regression: Acc=0.8123, F1=0.7183
✓ svm: Acc=0.8340, F1=0.7472
✓ random_forest: Acc=0.8641, F1=0.7982
✓ mlp: Acc=0.8461, F1=0.7783

✓ Model comparison chart saved to: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\evaluation\model_comparison.png


## 4. Tổng kết & Kết quả

Notebook này điều phối toàn bộ pipeline và xuất các artifacts cuối cùng phục vụ cho việc đánh giá và nộp bài:

- **Stage 1 — Data**: Thu thập và lưu trữ dữ liệu nguồn (`data/`).
- **Stage 2 — Preprocessing**: Imputation → Encoding → Scaling (lưu dataset đã xử lý tại `data/processed/`).
- **Stage 3 — Features**: Tạo và lưu đặc trưng (PCA hoặc các phương pháp khác) tại `features/`.
- **Stage 4 — Models**: Huấn luyện và lưu mô hình (Logistic, SVM, Random Forest, tùy chọn MLP).
- **Stage 5 — Evaluation**: Tính toán metrics, sinh đồ thị, và so sánh mô hình (lưu trong `reports/evaluation/`).

**Artifacts được tạo:**
- `data/processed/dataset_after_imputation.csv` — dữ liệu đã tiền xử lý
- `features/` — các file `.npy` hoặc `.h5` chứa đặc trưng
- `reports/eda/` — báo cáo EDA và hình ảnh
- `reports/evaluation/` — kết quả đánh giá, biểu đồ và bảng so sánh

Kết quả cuối cùng: chạy notebook này sẽ cho ra một bộ artifacts đầy đủ để kiểm tra và nộp sản phẩm.

In [15]:
# Display evaluation results summary table
print("\n" + "="*80)
print("FINAL RESULTS - MODEL PERFORMANCE SUMMARY")
print("="*80)

results_df = pd.DataFrame([
    {
        "Model": name,
        "Accuracy": f"{res['accuracy']:.4f}",
        "Precision": f"{res['precision']:.4f}",
        "Recall": f"{res['recall']:.4f}",
        "F1-Score": f"{res['f1']:.4f}"
    }
    for name, res in evaluation_results.items()
])
print(results_df.to_string(index=False))
print("="*80)

# Show artifact locations
print("\nArtefacts saved to:")
print(f"  EDA Reports:        {ROOT / CONFIG['eda']['output_dir']}")
print(f"  Features:           {ROOT / CONFIG['features']['output']['path']}")
print(f"  Evaluation Reports: {eval_output_dir}")
print(f"\nPipeline execution completed successfully! ✓")


FINAL RESULTS - MODEL PERFORMANCE SUMMARY
              Model Accuracy Precision Recall F1-Score
logistic_regression   0.8123    0.8089 0.6459   0.7183
                svm   0.8340    0.8572 0.6622   0.7472
      random_forest   0.8641    0.8875 0.7252   0.7982
                mlp   0.8461    0.8344 0.7292   0.7783

Artefacts saved to:
  EDA Reports:        c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\eda
  Features:           c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\features\processed_features
  Evaluation Reports: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\evaluation

Pipeline execution completed successfully! ✓


# Visualization — Biểu đồ trực quan hóa kết quả

Cell này tạo các biểu đồ: confusion matrix, ROC, Precision-Recall, model comparison, feature importance, và PCA 2D scatter. Các file ảnh được lưu vào thư mục đánh giá (`reports/evaluation/`).

In [16]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_curve, auc, precision_recall_curve, average_precision_score
from pathlib import Path

sns.set_style('whitegrid')

# Determine evaluation output directory
eval_dir = eval_output_dir if 'eval_output_dir' in globals() else (ROOT / CONFIG['evaluation']['output_dir'])
eval_dir.mkdir(parents=True, exist_ok=True)

# Prepare list for comparison metrics
comparison_metrics = []

for name, model_data in results.items():
    print(f"Processing visualizations for: {name}")
    # Confusion matrix
    try:
        y_pred = model_data['predictions']
        cm = confusion_matrix(y_test, y_pred)
        fig, ax = plt.subplots(figsize=(5,4))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
        ax.set_title(f'Confusion Matrix - {name}')
        ax.set_xlabel('Predicted'); ax.set_ylabel('True')
        p = eval_dir / f'{name}_confusion_matrix.png'
        fig.savefig(p, bbox_inches='tight'); plt.close(fig)
    except Exception as e:
        print(f'  ✗ Confusion matrix skipped for {name}: {e}')

    # ROC & Precision-Recall (binary only, if y_proba available)
    try:
        y_proba = model_data.get('y_proba', None)
        if y_proba is not None and len(np.unique(y_test)) == 2:
            fpr, tpr, _ = roc_curve(y_test, y_proba)
            roc_auc = auc(fpr, tpr)
            fig, ax = plt.subplots()
            ax.plot(fpr, tpr, label=f'AUC={roc_auc:.3f}')
            ax.plot([0,1],[0,1],'--', color='gray')
            ax.set_title(f'ROC - {name}'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.legend()
            p = eval_dir / f'{name}_roc.png'; fig.savefig(p, bbox_inches='tight'); plt.close(fig)

            precision, recall, _ = precision_recall_curve(y_test, y_proba)
            ap = average_precision_score(y_test, y_proba)
            fig, ax = plt.subplots()
            ax.plot(recall, precision, label=f'AP={ap:.3f}')
            ax.set_title(f'Precision-Recall - {name}'); ax.set_xlabel('Recall'); ax.set_ylabel('Precision'); ax.legend()
            p = eval_dir / f'{name}_pr.png'; fig.savefig(p, bbox_inches='tight'); plt.close(fig)
    except Exception as e:
        print(f'  ✗ ROC/PR skipped for {name}: {e}')

    # Collect comparison metrics if available
    if name in evaluation_results:
        try:
            comparison_metrics.append((name, evaluation_results[name]['accuracy'], evaluation_results[name]['f1']))
        except Exception:
            pass

# Model comparison bar chart
if comparison_metrics:
    names, accs, f1s = zip(*comparison_metrics)
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(8,4))
    width = 0.35
    ax.bar(x - width/2, accs, width, label='Accuracy')
    ax.bar(x + width/2, f1s, width, label='F1')
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=45)
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison (Accuracy vs F1)')
    ax.legend()
    p = eval_dir / 'model_comparison_scores.png'
    fig.savefig(p, bbox_inches='tight'); plt.close(fig)

# Feature importance (RF) and LR coefficients (top indices)
try:
    if 'random_forest' in results:
        rf = results['random_forest']['model']
        if hasattr(rf, 'feature_importances_'):
            fi = np.array(rf.feature_importances_)
            idx = np.argsort(fi)[::-1][:20]
            fig, ax = plt.subplots(figsize=(8,4))
            ax.bar(range(len(idx)), fi[idx])
            ax.set_xticks(range(len(idx)))
            ax.set_xticklabels([str(i) for i in idx], rotation=45)
            ax.set_title('Random Forest - Top feature importances (by index)')
            p = eval_dir / 'rf_feature_importances.png'
            fig.savefig(p, bbox_inches='tight'); plt.close(fig)
    if 'logistic_regression' in results:
        lr = results['logistic_regression']['model']
        if hasattr(lr, 'coef_'):
            coef = lr.coef_.ravel()
            idx = np.argsort(np.abs(coef))[::-1][:20]
            fig, ax = plt.subplots(figsize=(8,4))
            ax.bar(range(len(idx)), coef[idx])
            ax.set_xticks(range(len(idx)))
            ax.set_xticklabels([str(i) for i in idx], rotation=45)
            ax.set_title('Logistic Regression - Top coefficients (by index)')
            p = eval_dir / 'lr_top_coefficients.png'
            fig.savefig(p, bbox_inches='tight'); plt.close(fig)
except Exception as e:
    print('  ✗ Feature importance plotting failed:', e)

# PCA 2D scatter (train)
try:
    if 'X_train_pca' in globals() and getattr(X_train_pca, 'shape', (0,))[1] >= 2:
        fig, ax = plt.subplots(figsize=(6,5))
        scatter = ax.scatter(X_train_pca[:,0], X_train_pca[:,1], c=y_train, cmap='Spectral', alpha=0.6)
        ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.set_title('PCA 2D scatter (train)')
        fig.colorbar(scatter, ax=ax)
        p = eval_dir / 'pca_scatter_train.png'; fig.savefig(p, bbox_inches='tight'); plt.close(fig)
except Exception as e:
    print('  ✗ PCA scatter skipped:', e)

print('All visualization files (if created) are in:', eval_dir)


Processing visualizations for: logistic_regression
Processing visualizations for: svm
Processing visualizations for: random_forest
Processing visualizations for: mlp
All visualization files (if created) are in: c:\Users\Thanh Dung\Documents\MYDATA\BKU\4\2\ML\BTL\project_folder\reports\evaluation
